# Homework - Annotated Notebook

This notebook includes step-by-step markdown sections and English code comments.

## Step 1 - Import required libraries

In [1]:
# Import required libraries.
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

## Step 2 - Create a Spark session

In [2]:
# Create a Spark session.
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

22/03/07 21:55:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Step 3 - Run the next processing step

In [3]:
# Run the next processing step.
spark.version

'3.0.3'

## Step 4 - Inspect local files and raw data

In [4]:
# Inspect local files and raw data.
!ls -lh fhvhv_tripdata_2021-02.csv

-rw-rw-r-- 1 alexey alexey 700M Oct 29 18:53 fhvhv_tripdata_2021-02.csv


## Step 5 - Run the next processing step

In [6]:
# Run the next processing step.
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

## Step 6 - Load data into a Spark DataFrame

In [8]:
# Load data into a Spark DataFrame.
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-02.csv')

df = df.repartition(24)

df.write.parquet('data/pq/fhvhv/2021/02/', compression=)

## Step 7 - Load data into a Spark DataFrame

In [16]:
# Load data into a Spark DataFrame.
df = spark.read.parquet('data/pq/fhvhv/2021/02/')


[Stage 0:>                                                          (0 + 1) / 1]



**Q3**: How many taxi trips were there on February 15?

## Step 8 - Import required libraries

In [18]:
# Import required libraries.
from pyspark.sql import functions as F

## Step 9 - Run the next processing step

In [24]:
# Run the next processing step.
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .filter("pickup_date = '2021-02-15'") \
    .count()

367170

## Step 10 - Register a temporary SQL view

In [25]:
# Register a temporary SQL view.
df.registerTempTable('fhvhv_2021_02')

## Step 11 - Run a Spark SQL query

In [28]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    COUNT(1)
FROM 
    fhvhv_2021_02
WHERE
    to_date(pickup_datetime) = '2021-02-15';
""").show()

+--------+
|count(1)|
+--------+
|  367170|
+--------+




[Stage 20:==============>                                           (1 + 3) / 4]



**Q4**: Longest trip for each day

## Step 12 - Run the next processing step

In [29]:
# Run the next processing step.
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

## Step 13 - Run the next processing step

In [36]:
# Run the next processing step.
df \
    .withColumn('duration', df.dropoff_datetime.cast('long') - df.pickup_datetime.cast('long')) \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .groupBy('pickup_date') \
        .max('duration') \
    .orderBy('max(duration)', ascending=False) \
    .limit(5) \
    .show()

+-----------+-------------+
|pickup_date|max(duration)|
+-----------+-------------+
| 2021-02-11|        75540|
| 2021-02-17|        57221|
| 2021-02-20|        44039|
| 2021-02-03|        40653|
| 2021-02-19|        37577|
+-----------+-------------+




[Stage 38:==================================================>   (187 + 4) / 200]



## Step 14 - Run a Spark SQL query

In [38]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    to_date(pickup_datetime) AS pickup_date,
    MAX((CAST(dropoff_datetime AS LONG) - CAST(pickup_datetime AS LONG)) / 60) AS duration
FROM 
    fhvhv_2021_02
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 10;
""").show()

+-----------+-----------------+
|pickup_date|         duration|
+-----------+-----------------+
| 2021-02-11|           1259.0|
| 2021-02-17|953.6833333333333|
| 2021-02-20|733.9833333333333|
| 2021-02-03|           677.55|
| 2021-02-19|626.2833333333333|
| 2021-02-25|            583.5|
| 2021-02-18|576.8666666666667|
| 2021-02-10|569.4833333333333|
| 2021-02-21|           537.05|
| 2021-02-09|534.7833333333333|
+-----------+-----------------+




[Stage 44:================================================>     (180 + 4) / 200]



**Q5**: Most frequent `dispatching_base_num`

How many stages this spark job has?



## Step 15 - Run a Spark SQL query

In [44]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    dispatching_base_num,
    COUNT(1)
FROM 
    fhvhv_2021_02
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()


[Stage 73:>                                                         (0 + 4) / 4]



+--------------------+--------+
|dispatching_base_num|count(1)|
+--------------------+--------+
|              B02510| 3233664|
|              B02764|  965568|
|              B02872|  882689|
|              B02875|  685390|
|              B02765|  559768|
+--------------------+--------+




[Stage 74:===================================================>  (189 + 5) / 200]



## Step 16 - Run the next processing step

In [46]:
# Run the next processing step.
df \
    .groupBy('dispatching_base_num') \
        .count() \
    .orderBy('count', ascending=False) \
    .limit(5) \
    .show()


[Stage 86:>                                                         (0 + 4) / 4]



+--------------------+-------+
|dispatching_base_num|  count|
+--------------------+-------+
|              B02510|3233664|
|              B02764| 965568|
|              B02872| 882689|
|              B02875| 685390|
|              B02765| 559768|
+--------------------+-------+




[Stage 87:===========================================>          (161 + 5) / 200]



**Q6**: Most common locations pair

## Step 17 - Load data into a Spark DataFrame

In [47]:
# Load data into a Spark DataFrame.
df_zones = spark.read.parquet('zones')

## Step 18 - Run the next processing step

In [49]:
# Run the next processing step.
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

## Step 19 - Run the next processing step

In [51]:
# Run the next processing step.
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

## Step 20 - Register a temporary SQL view

In [50]:
# Register a temporary SQL view.
df_zones.registerTempTable('zones')

## Step 21 - Run a Spark SQL query

In [57]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    CONCAT(pul.Zone, ' / ', dol.Zone) AS pu_do_pair,
    COUNT(1)
FROM 
    fhvhv_2021_02 fhv LEFT JOIN zones pul ON fhv.PULocationID = pul.LocationID
                      LEFT JOIN zones dol ON fhv.DOLocationID = dol.LocationID
GROUP BY 
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()

+--------------------+--------+
|          pu_do_pair|count(1)|
+--------------------+--------+
|East New York / E...|   45041|
|Borough Park / Bo...|   37329|
| Canarsie / Canarsie|   28026|
|Crown Heights Nor...|   25976|
|Bay Ridge / Bay R...|   17934|
+--------------------+--------+



## Step 22 - Run this processing step

In [ ]:
# Run this processing step.
